# Absolute Time in Higher-Dimensional Geometry: a guided walkthrough

This notebook follows the paper *Absolute Time in Higher-Dimensional Geometry: Classically Simultaneous Relativity via an Unwrapped Kaluza–Klein Dimension* (A. Mueller) section by section. Each result is computed with the same functions that check the manuscript, from the `paper` subpackage of the **alpha-omega-gravity** package, and most of them are shown as figures.

**The idea in one sentence.** Relativistic effects are written as displacement along an additional, non-compact spatial coordinate $y$, whose slope is the Lorentz factor's partner, $\partial_x y=\beta\gamma$, so that $\gamma=\sqrt{1+(\partial_x y)^2}$. Time is not a dimension: it is the ratio of distances $\tau$, carried as the length $w=c\tau$.

**Route through the notebook**

1. The extra coordinate as a slope (special relativity)
2. Classical simultaneity in $\mathbb{R}^3$
3. The synchronization gauge
4. Gravity: the Flamm embedding and the flat absolute slice
5. One relation from planets to the Hubble radius
6. Interiors: uniform density, Buchdahl and Tolman VII
7. The scalar sector and the Cassini bound
8. Rotation: frame dragging and the Sagnac gap
9. Orbits (Appendix A of the paper)
10. Hypothesis M and the Milky Way
11. Hypothesis M and the SPARC sample
12. Co-scaling with $\gamma$: where the absolute frame could show
13. Dimensional audit and the full consistency dashboard
14. What is established, assumed and open

Throughout, the paper's own distinction is kept: where the construction is checked against general relativity it agrees with it, and those results are consistency checks, not new predictions. Sections 10 and 11 test the one explicit hypothesis that goes beyond it.

In [ ]:
# ---- Fill these in -------------------------------------------------------------
VIDEO_URL = ""  # YouTube link for the companion video (leave empty to skip)
PAPER_URL = "https://zenodo.org/records/24280888"
PYPI_URL = "https://pypi.org/project/alpha-omega-gravity/"

# If you are working from a local checkout instead of the PyPI package, set this to
# the folder that contains the top-level `physics` package (for example "~/code/physics").
LOCAL_PACKAGE_PATH = None

# Milky Way rotation-curve data used in section 10 (the file shipped with the paper).
ROTATION_CURVE_DATA = "../rc_results.npz"

# SPARC rotation curves used in section 11: a folder holding SPARC_Lelli2016c.mrt and Rotmod_LTG/.
# Download both from https://astroweb.case.edu/SPARC/ (unzip Rotmod_LTG.zip into Rotmod_LTG/).
SPARC_DATA_DIR = "~/sparc_cache"

# Audio walkthrough: the script is a plain text file you can edit. The audio is
# generated once from it (see the cell after the links) and then reused.
NARRATION_SCRIPT = "walkthrough_narration.txt"
NARRATION_AUDIO = "walkthrough_narration.m4a"  # or a .wav/.mp3 you recorded yourself
TTS_ENGINE = "auto"       # "auto", "say" (macOS built-in), "kokoro" (pip install kokoro soundfile)
TTS_VOICE = None          # e.g. "Samantha" for say, "af_heart" for kokoro; None = default

In [ ]:
# Uncomment to install the published package:
# %pip install alpha-omega-gravity

In [ ]:
import os, sys, re, math
if LOCAL_PACKAGE_PATH:
    sys.path.insert(0, os.path.expanduser(LOCAL_PACKAGE_PATH))
try:
    import physics.symbolic.function_generators.paper as aw          # the paper's functions
    from physics.symbolic.function_generators.paper import _constants as K  # SI constants
except ImportError as err:
    raise ImportError(
        "Could not import the paper subpackage. Install it with `pip install alpha-omega-gravity` "
        "(it is imported as `physics`), or set LOCAL_PACKAGE_PATH in the cell above."
    ) from err

import numpy as np
import sympy as smp
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from IPython.display import display, Math, Markdown

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11, "legend.frameon": False,
})
PALETTE = ["#2f5d8a", "#c0582b", "#4c9a5f", "#8a5aa8", "#b88a1e", "#5a5a5a"]

def lam(expr, *syms):
    "Turn a SymPy expression from the package into a NumPy function of `syms`."
    return smp.lambdify(syms, expr, "numpy")

def show(label, expr):
    display(Math(label + " = " + smp.latex(expr)))

# symbols used throughout (all quantities are lengths or ratios of lengths)
v, c, beta, x, w, tau, dx = smp.symbols("v c beta x w tau dx", positive=True)
R, Rs, L, Rstar = smp.symbols("R r_s L R_*", positive=True)
print("Loaded", sum(1 for n in dir(aw) if not n.startswith("_")), "paper functions.")

In [ ]:
if VIDEO_URL:
    from IPython.display import YouTubeVideo
    m = re.search(r"(?:v=|youtu\.be/|embed/)([A-Za-z0-9_-]{11})", VIDEO_URL)
    display(YouTubeVideo(m.group(1), width=720, height=405) if m else Markdown(f"[Companion video]({VIDEO_URL})"))
display(Markdown(f"Paper: [{PAPER_URL}]({PAPER_URL}) · Package: [{PYPI_URL}]({PYPI_URL})"))

### Audio walkthrough

Press play and scroll along. The narration names each section as it reaches it and takes about five minutes. The first run generates the audio from `walkthrough_narration.txt`; later runs reuse the file. To change the wording, edit the text file and delete the audio file.

In [ ]:
import shutil, subprocess
from pathlib import Path
from IPython.display import Audio

def make_narration(script=NARRATION_SCRIPT, out=NARRATION_AUDIO, engine=TTS_ENGINE, voice=TTS_VOICE):
    "Synthesize the narration once. Returns the path of the audio file, or None."
    script, out = Path(script), Path(out)
    for existing in (out, out.with_suffix(".wav")):
        if existing.exists():
            return existing
    text = script.read_text()
    use_say = engine == "say" or (engine == "auto" and shutil.which("say") and shutil.which("afconvert"))
    if use_say:                                   # macOS: no install needed
        aiff = out.with_suffix(".aiff")
        cmd = ["say", "-r", "175", "-f", str(script), "-o", str(aiff)] + (["-v", voice] if voice else [])
        subprocess.run(cmd, check=True)
        if out.suffix == ".m4a":
            subprocess.run(["afconvert", str(aiff), "-o", str(out), "-f", "m4af", "-d", "aac"], check=True)
            aiff.unlink()
        else:
            aiff.rename(out)
        return out
    try:                                          # cross-platform neural voice
        from kokoro import KPipeline
        import soundfile as sf
    except ImportError:
        print("No speech engine found. On macOS this uses the built-in `say` command; elsewhere run\n"
              "  pip install kokoro soundfile\n"
              f"or record yourself reading {script} and save it as {out}.")
        return None
    pipe = KPipeline(lang_code="a")
    chunks = [audio for _, _, audio in pipe(text, voice=voice or "af_heart", speed=1.0)]
    wav = out.with_suffix(".wav")
    sf.write(wav, np.concatenate(chunks), 24000)
    return wav

audio_path = make_narration()
if audio_path:
    display(Audio(filename=str(audio_path)))

## 1. The extra coordinate as a slope

For a body moving at speed $v$, the paper assigns a displacement $y$ along an additional spatial axis with

$$\partial_x y=\frac{v}{\sqrt{c^2-v^2}}=\beta\gamma .$$

The ordinary displacement $dx$, the extra displacement $dy$ and the extended displacement $dS=\gamma\,dx$ form a right triangle, so the Lorentz factor is the hypotenuse-to-base ratio,

$$\gamma=\sqrt{1+(\partial_x y)^2}.$$

With the rapidity $\eta$, $\gamma=\cosh\eta$ and $\partial_x y=\sinh\eta$: the triangle is the identity $\cosh^2\eta=1+\sinh^2\eta$.

In [ ]:
slope = aw.embedding_slope(v, c)
gam = aw.lorentz_gamma(v, c)
show(r"\partial_x y", slope)
show(r"\gamma", gam)
show(r"\left(1+(\partial_x y)^2\right)-\gamma^2", smp.simplify(aw.gamma_from_slope(slope)**2 - gam**2))

In [ ]:
slope_f = lam(aw.embedding_slope(beta, 1), beta)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.6))
for b, col in zip([0.3, 0.6, 0.85], PALETTE):
    s = slope_f(b)
    a1.plot([0, 1, 1, 0], [0, 0, s, 0], color=col, lw=2, label=fr"$\beta={b}$:  $\gamma={np.hypot(1, s):.3f}$")
    a1.annotate(fr"$dS=\gamma\,dx$", (0.45, s / 2 + 0.06), color=col, fontsize=9, rotation=np.degrees(np.arctan(s)))
a1.annotate(r"$dx$", (0.48, -0.12)); a1.annotate(r"$dy=\beta\gamma\,dx$", (1.03, 0.5))
a1.set_xlim(-0.05, 1.5); a1.set_ylim(-0.2, 1.8); a1.set_aspect("equal")
a1.set_title("Lorentz triangles: base $dx$, height $dy$, hypotenuse $dS$"); a1.legend(loc="upper left", fontsize=9)

eta = np.linspace(0, 2.2, 300)
a2.plot(np.sinh(eta), np.cosh(eta), color=PALETTE[0], lw=2.5, label=r"$\gamma=\sqrt{1+(\partial_x y)^2}$")
for b, col in zip([0.3, 0.6, 0.85, 0.95], PALETTE):
    a2.plot(slope_f(b), np.hypot(1, slope_f(b)), "o", color=col, ms=8, label=fr"$\beta={b}$")
a2.plot(np.sinh(eta), np.sinh(eta), "--", color="gray", lw=1, label=r"asymptote $\gamma\to\partial_x y$")
a2.set_xlabel(r"slope $\partial_x y=\sinh\eta$"); a2.set_ylabel(r"$\gamma=\cosh\eta$")
a2.set_title("The Lorentz factor is fixed by the slope alone"); a2.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 2. Classical simultaneity in $\mathbb{R}^3$

Label every event by an absolute position $x$ and the absolute time $\tau$, written as the length $w=c\tau$. An observer $\mathbf U$ moving at $v$ assigns each event the displacement

$$y_U(x)=\gamma\frac{v}{c}\,x,$$

and its ordinary, Einstein-synchronized clocks read

$$c\,t_U=\gamma\,c\tau-y_U(x),\qquad x_U=\gamma(x-v\tau).$$

These are exactly the Lorentz coordinates, for arbitrary events. The paper's Proposition (subsection 2.3) then follows:

* every observer recovers the **same** absolute time, $c\tau=(c\,t_U+y_U)/\gamma$;
* two events at the same $\tau$ have $c\,\Delta t_U=-\Delta y_U$: the relativity of simultaneity is a displacement along $y$, not a difference in time;
* a clock riding along its own worldline loses exactly its own advance along $y$, which gives $d\sigma=d\tau/\gamma$.

In [ ]:
X, T, V, C = smp.symbols("X T V C", positive=True)
clock = aw.observer_clock_reading(X, T, V, C)          # c t_U
yU = aw.y_offset(X, V, C)
show(r"c\,t_U", clock)
show(r"c\,t_U-\gamma(c\tau-\tfrac{v}{c}x)", smp.simplify(clock - aw.lorentz_pair(X, T, V, C)[1]))
show(r"\tau_{\rm recovered}-\tau", smp.simplify(aw.absolute_time_from_observer(clock, X, V, C) - T))

In [ ]:
clock_f = lam(aw.observer_clock_reading(X, T, V, 1), X, T, V)   # c = 1: lengths only
y_f = lam(aw.y_offset(X, V, 1), X, V)
tau_back_f = lam(aw.absolute_time_from_observer(smp.Symbol("ct"), X, V, 1), smp.Symbol("ct"), X, V)

xs = np.linspace(-3, 3, 200)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

ax = axes[0]  # spacetime diagram in lengths
b = 0.5
for w0 in np.arange(-2, 2.5, 1.0):
    ax.plot(xs, np.full_like(xs, w0), color=PALETTE[0], lw=1.6 if w0 == 0 else 0.8, alpha=1 if w0 == 0 else 0.5)
    ax.plot(xs, w0 / aw.lorentz_gamma(b, 1).evalf() + b * xs, color=PALETTE[1], lw=1.6 if w0 == 0 else 0.8,
            alpha=1 if w0 == 0 else 0.5, ls="--")
ax.plot(b * np.linspace(-3, 3, 10) * 0 + np.linspace(-1.5, 1.5, 10) * b, np.linspace(-1.5, 1.5, 10), color="k", lw=2, label=r"worldline of $\mathbf{U}$")
for xi in [1, 2, 3]:
    ax.annotate("", xy=(xi, b * xi), xytext=(xi, 0), arrowprops=dict(arrowstyle="<->", color=PALETTE[2]))
ax.plot([], [], color=PALETTE[0], label=r"absolute instants $w=c\tau$ = const")
ax.plot([], [], color=PALETTE[1], ls="--", label=r"$\mathbf{U}$'s 'simultaneous' lines $t_U$ = const")
ax.plot([], [], color=PALETTE[2], label=r"gap $= y_U/\gamma$")
ax.set_xlim(-3, 3.3); ax.set_ylim(-2.2, 2.2); ax.set_xlabel("absolute position $x$"); ax.set_ylabel(r"$w=c\tau$ (a length)")
ax.set_title(r"$v=0.5c$: tilted lines are clock settings, not time"); ax.legend(fontsize=8, loc="lower right")

ax = axes[1]  # clock readings along one absolute instant
for b, col in zip([0.2, 0.5, 0.8], PALETTE):
    ax.plot(xs, clock_f(xs, 0.0, b), color=col, lw=2, label=fr"$c\,t_U$, $v={b}c$")
    ax.plot(xs, -y_f(xs, b), ":", color="k", lw=1)
ax.plot([], [], ":", color="k", label=r"$-y_U(x)$ (same lines)")
ax.set_xlabel("absolute position $x$ at $\\tau=0$"); ax.set_ylabel("observer clock reading $c\\,t_U$")
ax.set_title(r"Along one instant, every clock offset is $-y_U$"); ax.legend(fontsize=9)

ax = axes[2]  # recovered absolute time
for b, col in zip([0.2, 0.5, 0.8], PALETTE):
    ax.plot(xs, tau_back_f(clock_f(xs, 0.0, b), xs, b), color=col, lw=3 - 0.8 * [0.2, 0.5, 0.8].index(b), label=fr"$v={b}c$")
ax.set_ylim(-1, 1); ax.set_xlabel("absolute position $x$"); ax.set_ylabel(r"recovered $c\tau=(c\,t_U+y_U)/\gamma$")
ax.set_title("Every observer recovers the same instant"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

The same holds for arbitrary events, not only along one instant. Below, 400 random events are read by four observers moving at different speeds. Their raw clock readings scatter widely, but after each adds its own $y_U$ and divides by $\gamma$, all four collapse onto the absolute time.

In [ ]:
rng = np.random.default_rng(7)
ev_x, ev_w = rng.uniform(-3, 3, 400), rng.uniform(-3, 3, 400)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
for b, col in zip([0.0, 0.3, 0.6, 0.9], PALETTE):
    ct = clock_f(ev_x, ev_w, b) if b else ev_w.copy()
    a1.scatter(ev_w, ct, s=6, color=col, alpha=0.6, label=fr"$v={b}c$")
    back = tau_back_f(ct, ev_x, b) if b else ct
    a2.scatter(ev_w, back, s=6, color=col, alpha=0.6, label=fr"$v={b}c$")
a1.set_title(r"Raw clock readings $c\,t_U$ disagree"); a2.set_title(r"Corrected by $y_U$: one absolute time")
for a in (a1, a2):
    a.set_xlabel(r"absolute $w=c\tau$"); a.legend(fontsize=9, markerscale=3)
a1.set_ylabel("value assigned by observer")
plt.tight_layout(); plt.show()

# chains of boosts: S -> U -> W equals S -> W at the composed velocity
v1, v2 = 0.6, 0.7
vw = float(aw.compose_velocity(v1, v2, 1))
print(f"Composed velocity of W relative to S: {vw:.6f} c")
xU = aw.lorentz_pair(1.3, 0.4, v1, 1)
xW = aw.lorentz_pair(xU[0], xU[1], v2, 1)
print("tau recovered by W through the chain:", float(aw.absolute_time_from_observer(xW[1], 1.3, vw, 1)), "(event had tau = 0.4)")

### What every observer agrees on

For two events at the same $\tau$, observer $\mathbf U$ measures the separation $\Delta x_U=\gamma\,\Delta x$ and the extra displacement $\Delta y_U=\gamma\beta\,\Delta x$. These are the hypotenuse and height of the triangle of section 1, so they change with the observer. The base, the **absolute separation in $\mathbb{R}^3$**, does not:

$$\Delta x^2=\Delta x_U^2-\Delta y_U^2 .$$

This is also why $G_{44}>0$ (a Euclidean embedding) is consistent with the interval: it is the same Pythagorean relation, read with the base as the invariant.

In [ ]:
B = smp.Symbol("B", positive=True)
base, height, hyp = aw.observer_triangle(1, B)
bs = np.linspace(0, 0.95, 200)
fig, ax = plt.subplots(figsize=(8, 4.3))
ax.plot(bs, lam(hyp, B)(bs), color=PALETTE[1], lw=2, label=r"$\Delta x_U=\gamma\,\Delta x$ (observer length)")
ax.plot(bs, lam(height, B)(bs), color=PALETTE[2], lw=2, label=r"$\Delta y_U=\gamma\beta\,\Delta x$")
ax.plot(bs, lam(hyp**2 - height**2, B)(bs) * np.ones_like(bs), color=PALETTE[0], lw=3, label=r"$\Delta x_U^2-\Delta y_U^2$ (absolute, invariant)")
ax.set_ylim(0, 3.5); ax.set_xlabel(r"observer speed $\beta$"); ax.set_ylabel(r"in units of $\Delta x$")
ax.set_title(r"Hypotenuse and height depend on the observer; the base in $\mathbb{R}^3$ does not"); ax.legend()
plt.tight_layout(); plt.show()

### Clock dynamics without inserting time dilation

Along its own worldline, $x=v\tau$, a clock advances along $y$ by $dy_U=\gamma\beta\,(v\,d\tau)$. Its reading is what remains of $\gamma\,c\,d\tau$ after that advance:

$$c\,d\sigma=\gamma\,c\,d\tau-dy_U=\frac{c\,d\tau}{\gamma}.$$

In [ ]:
total = lam(aw.lorentz_gamma(beta, 1), beta)(bs)
lost = lam(aw.y_offset(beta * 1, beta, 1), beta)(bs)          # dy per unit c dtau along x = v tau
kept = lam(aw.observer_clock_reading(beta * 1, 1, beta, 1), beta)(bs)
fig, ax = plt.subplots(figsize=(8, 4.3))
ax.fill_between(bs, 0, kept, color=PALETTE[0], alpha=0.75, label=r"clock reading $d\sigma/d\tau=1/\gamma$")
ax.fill_between(bs, kept, total, color=PALETTE[2], alpha=0.55, label=r"advance along $y$: $\gamma\beta^2$")
ax.plot(bs, total, color="k", lw=1.5, label=r"$\gamma$")
ax.plot(bs, np.sqrt(1 - bs**2), "w--", lw=1.2, label=r"$\sqrt{1-\beta^2}$ (check)")
ax.set_ylim(0, 3.3); ax.set_xlabel(r"$\beta$"); ax.set_ylabel(r"per unit $c\,d\tau$")
ax.set_title("A moving clock loses exactly its own displacement along $y$"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

## 3. The synchronization gauge

How far apart in $y$ two events are is the same as how far an observer's clock setting departs from the absolute one. Send light out and back from an observer moving at $u$ relative to the local flow. Against absolute time the two legs take different times, and Reichenbach's synchronization parameter comes out as

$$\epsilon=\tfrac12\left(1+\frac{u}{c}\right),\qquad |dy|=|2\epsilon-1|\,dS .$$

Einstein's convention is $\epsilon=\tfrac12$; the extra coordinate measures the departure from it.

In this geometry space flows **outward** around a mass, in the same direction as the Hubble flow, and a static observer stays at rest while space streams past it at $\beta c$. Relative to space, the static observer therefore moves inward, $u=-\beta c$, and

$$\epsilon_{\rm static}=\tfrac12\left(1-\beta\right).$$

Light sent outward rides with the flow at $c(1+\beta)$ and returns against it at $c(1-\beta)$. (The package's `FLOW_DIRECTION = +1` encodes this convention; infall, its time reverse, would give $\tfrac12(1+\beta)$.)

In [ ]:
u = smp.Symbol("u", positive=True)
show(r"\epsilon(u)", aw.radar_epsilon(u, c))
show(r"\epsilon_{\rm static\ observer\ in\ PG}(\beta)", aw.radar_epsilon_pg_static(beta))
eps_f = lam(aw.radar_epsilon_pg_static(beta), beta)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.3))
a1.plot(bs, eps_f(bs), color=PALETTE[0], lw=2.5, label=r"static observer, outward flow: $\epsilon=(1-\beta)/2$")
a1.axhline(0.5, color="gray", ls="--", label=r"Einstein, $\epsilon=1/2$")
a1.set_xlabel(r"flow speed $\beta$"); a1.set_ylabel(r"$\epsilon$"); a1.legend(loc="lower left"); a1.set_title("Radar synchronization against absolute time")

b0, Lm = 0.5, 1.0
t_out, t_back = Lm / (1 + b0), Lm / (1 - b0)          # with the outward flow, then against it
a2.plot([0, Lm], [0, t_out], color=PALETTE[1], lw=2, label=fr"outgoing, with the flow, at $c(1+\beta)$")
a2.plot([Lm, 0], [t_out, t_out + t_back], color=PALETTE[2], lw=2, label=fr"returning, against the flow, at $c(1-\beta)$")
a2.axhline(t_out, color=PALETTE[1], ls=":", lw=1); a2.axhline((t_out + t_back) / 2, color="gray", ls="--", lw=1)
a2.annotate("Einstein would set the\nreflection time here", (0.05, (t_out + t_back) / 2 + 0.06), fontsize=9)
a2.annotate(r"absolute reflection time", (0.05, t_out - 0.17), fontsize=9, color=PALETTE[1])
a2.set_xlabel("distance in the flat slice"); a2.set_ylabel(r"absolute $c\tau$"); a2.set_title(fr"$\beta={b0}$: the gap is the $y$-offset")
a2.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

## 4. Gravity: the Flamm embedding and the flat absolute slice

For a static spherical mass the flow speed is $\beta^2=2Gm(R)/(c^2R)$, which is $r_s/R$ outside. The slope of $y$ is $\beta\gamma$, and the graph of $y(R)$ is the **Flamm paraboloid**, $y=2\sqrt{r_s(R-r_s)}$. Its induced radial metric $1+(\partial_R y)^2=\gamma^2$ is the spatial part of the Schwarzschild metric.

The absolute slice is the Painlevé–Gullstrand slice $T=$ const, which is exactly flat. Space flows **outward** through it at $\beta c$, the local escape speed (about 11.2 km/s at Earth's surface), in the same direction as the Hubble flow. A static observer, such as someone standing on Earth, stays at rest while space streams past; the static radius itself does not change. A static observer's proper length $\gamma\,dR$ is the arc length along the Flamm graph above the flat base $dR$, the same triangle as in section 2. The clock rate of that observer (the lapse) is $N=1/\gamma$.

In [ ]:
beta2 = aw.schwarzschild_beta_squared(R, Rs)
y_flamm = aw.flamm_height_schwarzschild(R, Rs)
show(r"\beta^2", beta2); show(r"y(R)", y_flamm)
show(r"(\partial_R y)^2-(\beta\gamma)^2", smp.simplify(smp.diff(y_flamm, R)**2 - aw.flamm_slope(beta2)**2))
show(r"1+(\partial_R y)^2", smp.simplify(aw.induced_radial_metric(smp.diff(y_flamm, R))))
show(r"N=1/\gamma", aw.lapse_exterior(R, Rs))
show(r"{}^{(3)}\!R\ \text{of the PG slice}", aw.pg_spatial_ricci_scalar())

In [ ]:
yf = lam(y_flamm.subs(Rs, 1), R)
r = np.linspace(1.0, 9, 120); ph = np.linspace(0, 2 * np.pi, 120)
RR, PP = np.meshgrid(r, ph)
Z = yf(RR)
gam_r = 1 / np.sqrt(1 - 1 / RR)

fig = plt.figure(figsize=(13, 5.6))
ax = fig.add_subplot(1, 2, 1, projection="3d")
norm = colors.Normalize(vmin=1.0, vmax=2.5)
ax.plot_surface(RR * np.cos(PP), RR * np.sin(PP), Z, facecolors=cm.viridis(norm(np.clip(gam_r, 1.0, 2.5))),
                rstride=3, cstride=3, linewidth=0, antialiased=True, alpha=0.95)
ax.plot_surface(RR * np.cos(PP), RR * np.sin(PP), np.full_like(Z, -1.5), color="#d9d9d9", alpha=0.35, linewidth=0)
ax.text(0, -9.5, -1.4, "flat absolute slice (PG)", fontsize=9)
ax.set_title(r"Flamm graph $y(R)$ over the flat slice (colour: $\gamma$)")
ax.set_xlabel(r"$x/r_s$"); ax.set_ylabel(r"$z/r_s$"); ax.set_zlabel(r"$y/r_s$"); ax.view_init(24, -60)
m = cm.ScalarMappable(norm=norm, cmap="viridis"); fig.colorbar(m, ax=ax, shrink=0.6, label=r"$\gamma$ (clipped at 2.5)")

ax = fig.add_subplot(1, 2, 2)
rr = np.linspace(1.001, 12, 600)
b2 = 1 / rr
ax.plot(rr, b2, color=PALETTE[0], lw=2, label=r"$\beta^2=r_s/R$")
ax.plot(rr, np.sqrt(b2 / (1 - b2)), color=PALETTE[1], lw=2, label=r"slope $\partial_R y=\beta\gamma$")
ax.plot(rr, 1 / np.sqrt(1 - b2), color=PALETTE[2], lw=2, label=r"$\gamma$ (length per $dR$)")
ax.plot(rr, np.sqrt(1 - b2), color=PALETTE[3], lw=2, label=r"lapse $N=1/\gamma$ (clock rate)")
ax.set_ylim(0, 3); ax.set_xlabel(r"$R/r_s$"); ax.set_title("Radial profiles outside a mass"); ax.legend()
plt.tight_layout(); plt.show()

### The synchronization gap *is* the Flamm height

Here $y$ is not defined from a Lorentz transformation. It is fixed by the spatial geometry. Independently, with space flowing outward, the offset between absolute (PG) time $T$ and static time $t$ is $c\,(dT-dt)=-\beta\,dR/(1-\beta^2)$: the absolute clocks run behind the static ones as one moves inward. The two coincide:

$$c\,(dT-dt)=-\gamma\,dy .$$

Below, the left side is integrated from the PG time shift and the right side from a numerical derivative of the Flamm height. They are computed separately and lie on top of each other.

In [ ]:
shift_f = lam(aw.pg_time_shift(smp.sqrt(beta2.subs(Rs, 1)), 1), R)           # c (dT - dt)/dR
rr = np.linspace(1.05, 30, 4000)
gap = np.cumsum(shift_f(rr)[::-1] * np.gradient(rr)[::-1])[::-1]               # integrate inward from R = 30 r_s
dy_dR = np.gradient(yf(rr), rr)
gam_dy = np.cumsum((1 / np.sqrt(1 - 1 / rr) * dy_dR)[::-1] * np.gradient(rr)[::-1])[::-1]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.3))
a1.plot(rr, -gap, color=PALETTE[0], lw=4, alpha=0.6, label=r"$-\int c\,(dT-dt)$ from the PG time shift")
a1.plot(rr, gam_dy, "--", color=PALETTE[1], lw=2, label=r"$\int\gamma\,dy$ from the Flamm height")
a1.set_xlabel(r"$R/r_s$"); a1.set_ylabel(r"accumulated offset (units of $r_s$)"); a1.legend(); a1.set_xlim(1, 30)
a1.set_title("Synchronization gap vs embedding height")
a2.semilogy(rr, np.abs(-gap - gam_dy) / np.abs(gap), color=PALETTE[2])
a2.set_xlabel(r"$R/r_s$"); a2.set_ylabel("relative difference"); a2.set_title("Difference (numerical integration error only)")
plt.tight_layout(); plt.show()

## 5. One relation from planets to the Hubble radius

The same expression $\beta^2=2Gm(R)/(c^2R)$ describes the flow around a mass and, with the mass of a uniform universe inside $R$, the Hubble flow $\beta_H=HR/c$. Both flows point outward, so around a mass embedded in the expanding universe they are one flow, $\beta^2=\beta_N^2+\beta_H^2$, whose static form is the Schwarzschild–de Sitter metric. At the Hubble radius $R_H=c/H$ the flow reaches $\beta_H=1$, which is the Sciama identity

$$\frac{G\,m_H(R_H)}{c^2R_H}=\frac12 .$$

In [ ]:
H = smp.Symbol("H", positive=True); G_ = smp.Symbol("G", positive=True)
show(r"\frac{G m_H(R_H)}{c^2 R_H}", aw.sciama_ratio(H, G_, c))
show(r"\beta_H^2-\frac{2 G m_H}{c^2R}", smp.simplify(aw.hubble_beta(H, R, c)**2 - aw.hubble_beta_squared_from_mass(H, R, G_, c)))

bodies = {                     # (GM [m^3/s^2], R [m])
    "Earth surface": (K.GM_EARTH, K.R_EARTH),
    "Saturn": (K.GM_SATURN, K.R_SATURN),
    "Sun surface": (K.GM_SUN, K.R_SUN),
    "Earth orbit": (K.GM_SUN, 1.495978707e11),
    "white dwarf (Sirius B)": (1.018 * K.GM_SUN, 5.8e6),
    "neutron star": (1.4 * K.GM_SUN, 1.2e4),
    "Sun in the Galaxy": (K.G * 4.44e10 * K.M_SUN, 8.12 * K.KPC),
}
RH = K.C / K.H0
Rgrid = np.logspace(3, np.log10(RH) + 0.3, 400)

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.loglog(Rgrid, (K.H0 * Rgrid / K.C) ** 2, color=PALETTE[0], lw=2.5, label=r"Hubble flow $\beta_H^2=(HR/c)^2$")
for (name, (GM, Rb)), col in zip(bodies.items(), PALETTE * 2):
    b2b = 2 * GM / (K.C**2 * Rb)
    ax.loglog(Rb, b2b, "o", ms=9, color=col)
    ax.annotate(name, (Rb, b2b), xytext=(6, 4), textcoords="offset points", fontsize=9)
ax.axvline(RH, color="gray", ls="--"); ax.axhline(1, color="gray", ls=":")
ax.annotate(r"$R_H=c/H_0$: $\beta_H=1$, $Gm_H/(c^2R_H)=1/2$", (RH, 1), xytext=(-300, 12), textcoords="offset points", fontsize=9)
ax.set_xlabel("radius $R$ [m]"); ax.set_ylabel(r"$\beta^2=2Gm/(c^2R)$"); ax.set_ylim(1e-14, 300)
ax.set_title("One dimensionless flow, from planets to the cosmic horizon"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

## 6. Interiors: uniform density, the Buchdahl bound and Tolman VII

Inside matter, $\beta^2=2Gm(R)/(c^2R)$ still fixes the slope. For uniform density, $\beta^2=R^2/L^2$ and the slice is a cap of a 3-sphere of radius $L$, which joins the exterior Flamm graph smoothly. The lapse obeys the TOV equation, and the central pressure diverges when the star is compressed to $R_*=\tfrac{9}{8} r_s$ (the Buchdahl bound).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
ax = axes[0]
for Rst, col in zip([4.0, 2.0, 1.3], PALETTE):
    Lst = np.sqrt(Rst**3)                                  # L^2 = R*^3 / r_s  (r_s = 1)
    rin = np.linspace(0, Rst, 200); rout = np.linspace(Rst, 10, 300)
    y_out = yf(rout)
    y_in = -np.sqrt(Lst**2 - rin**2) + np.sqrt(Lst**2 - Rst**2) + yf(Rst)
    ax.plot(np.r_[-rout[::-1], -rin[::-1], rin, rout], np.r_[y_out[::-1], y_in[::-1], y_in, y_out], color=col, lw=2,
            label=fr"$R_*={Rst}\,r_s$")
    ax.plot([-Rst, Rst], [yf(Rst)] * 2, "o", color=col, ms=4)
ax.set_xlabel(r"$R/r_s$"); ax.set_ylabel(r"$y/r_s$"); ax.set_title("Interior cap joined to the Flamm graph"); ax.legend(fontsize=9)

ax = axes[1]
lapse_f = lam(aw.uniform_density_lapse(R, Rstar, L), R, Rstar, L)
for Rst, col in zip([4.0, 2.0, 1.3, 1.15], PALETTE):
    Lst = np.sqrt(Rst**3); rin = np.linspace(0, Rst, 200); rout = np.linspace(Rst, 6, 200)
    ax.plot(np.r_[rin, rout], np.r_[lapse_f(rin, Rst, Lst), np.sqrt(1 - 1 / rout)], color=col, lw=2, label=fr"$R_*={Rst}\,r_s$")
ax.set_xlabel(r"$R/r_s$"); ax.set_ylabel("lapse $N$ (clock rate)"); ax.set_title("Clock rate through the star"); ax.legend(fontsize=9)

ax = axes[2]
s_star = lambda Rst: np.sqrt(1 - 1 / Rst)
pc = lam(aw.central_pressure_ratio(smp.Symbol("s")), smp.Symbol("s"))
Rg = np.linspace(1.1255, 4, 500)
ax.semilogy(Rg, pc(s_star(Rg)), color=PALETTE[0], lw=2.5)
ax.axvline(float(aw.buchdahl_radius_ratio(1, Rstar)), color=PALETTE[1], ls="--", label=r"Buchdahl $R_*=\frac{9}{8} r_s$")
ax.set_xlabel(r"$R_*/r_s$"); ax.set_ylabel(r"$p_c/\rho c^2$"); ax.set_title("Central pressure of a uniform star"); ax.legend()
plt.tight_layout(); plt.show()

A more realistic profile is Tolman VII, $\rho=\rho_c(1-x^2)$ with $x=R/R_*$. Its slice stays embeddable for compactness $C=r_s/R_*<24/25$, but the pressure diverges earlier, near $C\approx0.77$. The package integrates the TOV system inward from the surface.

In [ ]:
xg = np.linspace(0, 1, 300)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
Cs = [0.2, 0.4, 0.6, 0.7, 0.75]
b2_tvii = lam(aw.tolman_vii_beta_squared(x, smp.Symbol("C")), x, smp.Symbol("C"))
for Cv, col in zip(Cs, PALETTE):
    sol = aw.tolman_vii_integrate(Cv)
    axes[0].plot(xg, b2_tvii(xg, Cv), color=col, lw=2, label=fr"$C={Cv}$")
    axes[1].semilogy(sol["x"], np.maximum(sol["Pi"], 1e-6), color=col, lw=2, label=fr"$C={Cv}$: $\Pi_c={sol['Pi_c']:.2f}$")
    axes[2].plot(sol["x"], sol["N"], color=col, lw=2, label=fr"$C={Cv}$: $N_c={sol['N_c']:.3f}$")
axes[0].axhline(1, color="gray", ls=":"); axes[0].set_title(r"$\beta^2(x)$ (embedding needs $<1$)")
axes[1].set_title(r"pressure $\Pi=p/\rho_c c^2$"); axes[1].set_ylim(1e-4, 10)
axes[2].set_title("lapse $N$ (clock rate)")
for a in axes: a.set_xlabel(r"$x=R/R_*$"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

Cscan = np.linspace(0.05, 0.79, 60)
Pic = [aw.tolman_vii_integrate(Cv, n_steps=2000)["Pi_c"] for Cv in Cscan]
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.semilogy(Cscan, Pic, color=PALETTE[0], lw=2.5)
ax.axvline(24 / 25, color=PALETTE[1], ls="--", label="embedding bound $C=24/25$")
ax.set_xlim(0, 1); ax.set_xlabel("compactness $C=r_s/R_*$"); ax.set_ylabel(r"central $\Pi_c$")
ax.set_title("Tolman VII: the pressure diverges before the embedding fails"); ax.legend()
plt.tight_layout(); plt.show()

## 7. The scalar sector and the Cassini bound

If the size of the $y$ direction, $G_{44}=\Phi^2$, is allowed to vary, the five-dimensional equations reduce to a Brans–Dicke theory with $\omega_{\rm BD}=0$. Such a scalar would bend light by $\gamma_{\rm PPN}=1/2$ of the general-relativistic amount, which the Cassini measurement excludes ($\omega_{\rm BD}\gtrsim4\times10^4$).

General relativity, and Cassini, are recovered exactly when the stress along $y$ takes the value

$$p_y=\frac{3p-\epsilon}{2}\qquad(-\epsilon/2\text{ for dust}),$$

because then the scalar has no source and $\Phi$ stays constant. These results do not depend on the sign of $G_{44}$ (`y_signature.py`); the positive sign is required by the Flamm embedding and by the sign of the field-strength term of the four-potential.

In [ ]:
om = smp.Symbol("omega", positive=True)
show(r"\gamma_{\rm PPN}(\omega_{\rm BD})", aw.brans_dicke_gamma_ppn(om))
show(r"\omega_{\rm BD}\ \text{(cylinder sector)}", aw.cylinder_sector_omega())
eps_m, p = smp.symbols("epsilon p")
show(r"p_y^{\rm (GR)}", aw.gr_recovery_py(eps_m, p))
show(r"\text{scalar source at }p_y^{\rm (GR)}", smp.simplify(aw.scalar_source_box_phi(eps_m, p, aw.gr_recovery_py(eps_m, p), 1, 1)))
omega_min = aw.omega_bd_lower_bound(K.CASSINI_SIGMA)
print(f"Cassini: gamma - 1 = ({K.CASSINI_GAMMA_MINUS_1:.1e} +- {K.CASSINI_SIGMA:.1e})  ->  omega_BD > {omega_min:.1e} (the usual 1-sigma quote)")

omg = np.logspace(-1, 6, 400)
gp = lam(aw.brans_dicke_gamma_ppn(om), om)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
a1.loglog(omg, 1 - gp(omg), color=PALETTE[0], lw=2.5, label=r"Brans–Dicke $1-\gamma_{\rm PPN}=1/(2+\omega)$")
a1.axhspan(1e-7, K.CASSINI_SIGMA, color=PALETTE[2], alpha=0.25, label=r"allowed by Cassini ($|\gamma-1|<2.3\times10^{-5}$)")
a1.plot(0.1, 0.5, ">", color=PALETTE[1], ms=12, label=r"cylinder sector, $\omega_{\rm BD}=0$ ($\gamma=1/2$)")
a1.axvline(omega_min, color=PALETTE[2], ls="--")
a1.set_xlabel(r"$\omega_{\rm BD}$"); a1.set_ylabel(r"$1-\gamma_{\rm PPN}$"); a1.set_ylim(1e-7, 1); a1.legend(fontsize=9)
a1.set_title("A freely varying $G_{44}$ is excluded")

w_eos = np.linspace(0, 1 / 3, 100)
a2.plot(w_eos, (3 * w_eos - 1) / 2, color=PALETTE[0], lw=2.5, label=r"$p_y/\epsilon=(3p/\epsilon-1)/2$")
a2.plot(0, -0.5, "o", color=PALETTE[1], ms=8); a2.annotate("dust: $p_y=-\\epsilon/2$", (0, -0.5), xytext=(12, 4), textcoords="offset points")
a2.plot(1 / 3, 0, "o", color=PALETTE[2], ms=8); a2.annotate("radiation: $p_y=0$", (1 / 3, 0), xytext=(-120, -4), textcoords="offset points")
a2.set_ylim(-0.6, 0.1)
a2.set_xlabel(r"equation of state $p/\epsilon$"); a2.set_ylabel(r"required $p_y/\epsilon$"); a2.legend(loc="upper left")
a2.set_title("Stress along $y$ that recovers GR")
plt.tight_layout(); plt.show()

## 8. Rotation: frame dragging and the Sagnac gap

A rotating source adds a non-gradient (curl) part to the flow. On the flat slice the momentum constraint gives $(R^4\omega')'=0$, so $\omega=2GJ/(c^2R^3)$, the Lense–Thirring result, and nothing else (a $1/R^2$ profile fails the constraint). For Gravity Probe B's orbit this gives about 41 mas/yr against the measured $37.2\pm7.2$.

The same non-gradient part appears in a rotating frame as a synchronization gap that does not close around a loop: the Sagnac effect, $\Delta t=2\pi\Omega r^2\gamma^2/c^2$, about 207 ns around the equator.

In [ ]:
Jsym, Gs = smp.symbols("J G", positive=True)
show(r"\omega_{\rm LT}", aw.lense_thirring_omega(Jsym, R, Gs, c))
show(r"(R^4\omega')'", aw.momentum_constraint_residual(aw.lense_thirring_omega(Jsym, R, Gs, c), R))
omega_s, polar = aw.gravitomagnetic_numbers()
print(f"Earth surface omega = {omega_s:.3e} rad/s;  GP-B polar-orbit average = {polar:.1f} mas/yr")

Om, rr_s = smp.symbols("Omega r", positive=True)
sag = aw.sagnac_loop_gap(Om, c, rr_s)
show(r"\oint g_\phi\,d\phi", sag)
sag_f = lam(sag, Om, c, rr_s)
OMEGA_E = 7.2921150e-5
lat = np.linspace(0, 90, 200)

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4.3))
Rr = np.linspace(1, 4, 200) * K.R_EARTH
lt = lam(aw.lense_thirring_omega(Jsym, R, Gs, c), Jsym, R, Gs, c)
a1.plot(Rr / K.R_EARTH, lt(K.J_EARTH, Rr, K.G, K.C) * K.YEAR * K.MAS_PER_RAD, color=PALETTE[0], lw=2.5)
a1.set_xlabel(r"$R/R_\oplus$"); a1.set_ylabel(r"$\omega_{\rm LT}$ [mas/yr]"); a1.set_title(r"Frame dragging $\propto R^{-3}$")

a2.bar(["this construction\n(= GR)", "Gravity Probe B"], [polar, 37.2], color=[PALETTE[0], PALETTE[1]])
a2.errorbar([1], [37.2], yerr=[7.2], fmt="none", ecolor="k", capsize=8)
a2.text(0, polar + 1, f"{polar:.1f}", ha="center"); a2.text(1.28, 38, "37.2 ± 7.2", ha="left"); a2.set_xlim(-0.6, 1.9)
a2.set_ylabel("frame-dragging precession [mas/yr]"); a2.set_title("Gyroscope precession, polar orbit")

a3.plot(lat, sag_f(OMEGA_E, K.C, K.R_EARTH * np.cos(np.radians(lat))) * 1e9, color=PALETTE[2], lw=2.5)
a3.set_xlabel("latitude [deg]"); a3.set_ylabel("loop synchronization gap [ns]")
a3.set_title(f"Sagnac gap: {sag_f(OMEGA_E, K.C, K.R_EARTH) * 1e9:.1f} ns at the equator")
plt.tight_layout(); plt.show()

## 9. Orbits (Appendix A of the paper)

With the induced radial metric $A=1+(\partial_R y)^2$ and the lapse $N^2=1/A$, geodesic motion is that of the Schwarzschild metric. The effective potential below is built from the package's embedding functions, and its innermost stable circular orbit and perihelion advance are the standard ones.

In [ ]:
r_, Lang = smp.symbols("r L", positive=True)
A = aw.induced_radial_metric(aw.flamm_slope(aw.schwarzschild_beta_squared(r_, 1)))   # units G = M = c = 1, r_s = 2 -> use r_s = 1 here
N2 = smp.simplify(1 / A)
Veff = N2 * (1 + Lang**2 / r_**2)
show(r"N^2", N2); show(r"V_{\rm eff}", Veff)
isco = smp.solve([smp.diff(Veff, r_), smp.diff(Veff, r_, 2)], [r_, Lang], dict=True)
print("ISCO (in units of r_s):", [s for s in isco if s[r_].is_positive])

V_f = lam(Veff, r_, Lang)
rg = np.linspace(1.05, 20, 800)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
for Lv, col in zip([1.6, np.sqrt(3), 2.0, 2.3], PALETTE):
    a1.plot(rg, V_f(rg, Lv), color=col, lw=2, label=fr"$L={Lv:.3f}\,r_s c$")
a1.axvline(3, color="gray", ls="--", label=r"ISCO $=3r_s$")
a1.set_ylim(0.8, 1.15); a1.set_xlabel(r"$r/r_s$"); a1.set_ylabel(r"$V_{\rm eff}=N^2(1+L^2/r^2)$"); a1.legend(fontsize=9)
a1.set_title("Effective potential from the embedding")

planets = {"Mercury": (5.7909e10, 0.2056, 87.969, 42.98), "Venus": (1.0821e11, 0.0068, 224.701, 8.62),
           "Earth": (1.496e11, 0.0167, 365.256, 3.84), "Mars": (2.2794e11, 0.0934, 686.98, 1.35)}
model = {n: 6 * np.pi * K.GM_SUN / (K.C**2 * a * (1 - e**2)) * (36525 / P) * 206264.806 for n, (a, e, P, _) in planets.items()}
xi = np.arange(len(planets))
a2.bar(xi - 0.2, list(model.values()), 0.4, color=PALETTE[0], label="this construction (= GR)")
a2.bar(xi + 0.2, [p[3] for p in planets.values()], 0.4, color=PALETTE[1], label="GR value in the literature")
for i, (mv, lv) in enumerate(zip(model.values(), [p[3] for p in planets.values()])):
    a2.text(i - 0.2, mv * 1.08, f"{mv:.2f}", ha="center", fontsize=8); a2.text(i + 0.2, lv * 1.08, f"{lv:.2f}", ha="center", fontsize=8)
a2.set_xticks(xi, list(planets)); a2.set_yscale("log"); a2.set_ylim(1, 80); a2.set_ylabel("perihelion advance [arcsec/century]")
a2.set_title("Relativistic perihelion advance"); a2.legend()
plt.tight_layout(); plt.show()
print({k: round(v, 2) for k, v in model.items()})

## 10. Hypothesis M and the Milky Way

This is the one part of the paper that goes beyond general relativity, and it is stated as a hypothesis. Inertial resistance is assumed to be cut off at the cosmic horizon, which gives a characteristic acceleration

$$g_0=\kappa'\,cH_0,$$

with the observed acceleration following the radial-acceleration relation $g=g_N/(1-e^{-\sqrt{g_N/g_0}})$. The candidate $\kappa'=1/2\pi$ gives $g_0\approx1.1\times10^{-10}$ m/s². In the deep limit this gives the baryonic Tully–Fisher relation $v^4=GMg_0$.

The rotation-curve data below are those used in the paper (Eilers et al. 2019, with the rescaled baryonic model, from `rc_results.npz`).

In [ ]:
g0_c = aw.g0_candidate()
rar = np.vectorize(aw.rar_acceleration)
gN = np.logspace(-13, -8, 300)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6))
a1.loglog(gN, rar(gN, g0_c), color=PALETTE[0], lw=2.5, label=r"Hypothesis M, $\kappa'=1/2\pi$")
a1.loglog(gN, gN, "--", color="gray", label="Newton, $g=g_N$")
a1.loglog(gN, np.sqrt(gN * g0_c), ":", color=PALETTE[1], lw=2, label=r"deep limit $\sqrt{g_N g_0}$")
a1.axvline(g0_c, color=PALETTE[2], lw=1); a1.annotate(r"$g_0=cH_0/2\pi$", (g0_c, 1e-12), xytext=(5, 0), textcoords="offset points")
a1.set_xlabel(r"baryonic $g_N$ [m s$^{-2}$]"); a1.set_ylabel(r"observed $g$ [m s$^{-2}$]"); a1.legend(fontsize=9)
a1.set_title("Radial-acceleration relation")

Mb = np.logspace(8, 12, 100) * K.M_SUN
a2.loglog(Mb / K.M_SUN, np.vectorize(aw.btfr_velocity)(K.G * Mb, g0_c) / 1e3, color=PALETTE[0], lw=2.5, label=r"$v_f=(GMg_0)^{1/4}$")
a2.set_yticks([30, 50, 100, 200, 300], ["30", "50", "100", "200", "300"])
a2.set_xlabel(r"baryonic mass [$M_\odot$]"); a2.set_ylabel(r"flat rotation speed [km/s]"); a2.legend()
a2.set_title("Baryonic Tully–Fisher slope 4 (no per-galaxy parameter)")
plt.tight_layout(); plt.show()

In [ ]:
d = np.load(ROTATION_CURVE_DATA)
Rk, vobs, sig = d["R"], d["v"], d["sig"]
Rm = Rk * K.KPC
gN_mw = (d["v_newt"] * 1e3) ** 2 / Rm
cH0 = K.C * K.H0

def v_hyp(kappa):
    return np.sqrt(Rm * rar(gN_mw, kappa * cH0)) / 1e3

# The paper fits the 32 points with errors below 10 km/s (the six outermost radii are excluded).
sel = sig < 10
kgrid = np.linspace(0.05, 0.8, 751)

def chi2_of(kappa, mask):
    return np.sum(((vobs[mask] - v_hyp(kappa)[mask]) / sig[mask]) ** 2)

chi2 = np.array([chi2_of(k, sel) for k in kgrid])
kbest = kgrid[np.argmin(chi2)]
dof = sel.sum() - 1
chi2_all = np.array([chi2_of(k, np.ones_like(sel)) for k in kgrid])
print(f"{sel.sum()} points with sigma < 10 km/s: best fit g0/(c H0) = {kbest:.3f}, chi2/dof = {chi2.min()/dof:.2f}  (paper: 0.35, 5.2)")
print(f"all {len(sig)} points: best fit g0/(c H0) = {kgrid[np.argmin(chi2_all)]:.3f}, chi2/dof = {chi2_all.min()/(len(sig)-1):.2f}")
print(f"candidate 1/2pi = {1/(2*np.pi):.3f}: chi2/dof = {chi2_of(1/(2*np.pi), sel)/dof:.2f} on the 32 points")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.6, 1]})
a1.errorbar(Rk[sel], vobs[sel], sig[sel], fmt="o", color="k", ms=4, capsize=2, label="Eilers et al. 2019 (fitted, $\\sigma<10$ km/s)")
a1.errorbar(Rk[~sel], vobs[~sel], sig[~sel], fmt="o", mfc="white", color="gray", ms=4, capsize=2, label="excluded from the fit")
a1.plot(Rk, d["v_newt"], color="gray", lw=2, ls="--", label="Newtonian, baryons only")
a1.plot(Rk, d["v_pap"], color=PALETTE[1], lw=2, label=r"flow velocity $v_{\rm orbit}=\beta c$")
a1.plot(Rk, v_hyp(1 / (2 * np.pi)), color=PALETTE[0], lw=2.5, label=r"Hypothesis M, $\kappa'=1/2\pi$")
a1.plot(Rk, v_hyp(kbest), color=PALETTE[2], lw=2, ls="-.", label=fr"Hypothesis M, best fit $\kappa'={kbest:.2f}$")
a1.set_xlabel("Galactocentric radius [kpc]"); a1.set_ylabel("circular speed [km/s]"); a1.set_ylim(100, 280)
a1.set_title("Milky Way rotation curve (adopted baryonic mass)"); a1.legend(fontsize=9, loc="lower left")
a2.plot(kgrid, chi2 / dof, color=PALETTE[0], lw=2.5)
a2.axvline(1 / (2 * np.pi), color=PALETTE[1], ls="--", label=r"$1/2\pi$")
a2.axvline(kbest, color=PALETTE[2], ls="-.", label="best fit")
a2.set_yscale("log"); a2.set_xlabel(r"$\kappa'=g_0/(cH_0)$"); a2.set_ylabel(r"$\chi^2/$dof (32 points)"); a2.legend()
a2.set_title("One galaxy cannot fix $\\kappa'$")
plt.tight_layout(); plt.show()

The paper's reading of this figure: the flow velocity alone falls too fast, Hypothesis M has the right shape, and the candidate $\kappa'=1/2\pi$ is within a factor of about two of the single-galaxy fit. Because the baryonic mass of the Milky Way is uncertain, this is consistent with the hypothesis but is not a test of it. A test needs a sample of galaxies with measured baryonic masses, which section 11 provides.

## 11. Hypothesis M and the SPARC sample

The SPARC database (Lelli, McGaugh & Schombert 2016) has rotation curves and baryonic mass models for 175 disc galaxies. With the cuts of the radial-acceleration-relation analysis, quality $Q<3$ and inclination $i\geq30^\circ$, 153 galaxies and 3168 points remain.

For each galaxy the stellar mass-to-light ratio ($0.5$ for the disc, $0.7$ for the bulge, $0.1$ dex scatter), the distance and the inclination are marginalized over their catalogue uncertainties. One $\kappa'$ is then fitted to all galaxies together, and its uncertainty comes from resampling the galaxies. The fit is run twice: with the quoted velocity errors, and with a $5\%$ floor for systematics such as non-circular motions.

In [ ]:
from pathlib import Path
cH0 = K.C * K.H0
sparc_dir = Path(os.path.expanduser(SPARC_DATA_DIR))
HAVE_SPARC = (sparc_dir / "SPARC_Lelli2016c.mrt").exists() and (sparc_dir / "Rotmod_LTG").is_dir()
if HAVE_SPARC:
    gals = aw.load_sparc(sparc_dir)
    print(f"{len(gals)} galaxies and {sum(len(g.R) for g in gals)} points after the cuts Q < 3, i >= 30 deg")
else:
    print(f"SPARC data not found in {sparc_dir}. Download SPARC_Lelli2016c.mrt and Rotmod_LTG.zip from "
          "https://astroweb.case.edu/SPARC/ into that folder, unzip the rotation curves into Rotmod_LTG/, "
          "and rerun this section. The cells below are skipped until then.")

In [ ]:
if HAVE_SPARC:
    from physics.symbolic.function_generators.paper import sparc as sp
    ups_d, ups_b = sp.UPSILON_DISK, sp.UPSILON_DISK * sp.UPSILON_BULGE_RATIO
    gb, go = [], []
    for g in gals:
        Rm = g.R * K.KPC
        vb2 = (np.abs(g.Vgas) * g.Vgas + ups_d * np.abs(g.Vdisk) * g.Vdisk + ups_b * np.abs(g.Vbul) * g.Vbul) * 1e6
        ok = vb2 > 0
        gb.append(vb2[ok] / Rm[ok]); go.append((g.Vobs[ok] * 1e3) ** 2 / Rm[ok])
    lgb, lgo = np.log10(np.concatenate(gb)), np.log10(np.concatenate(go))

    fig, ax = plt.subplots(figsize=(7.5, 6.2))
    hb = ax.hexbin(lgb, lgo, gridsize=55, cmap="Blues", mincnt=1, bins="log", extent=(-12.5, -8.5, -12.5, -8.5))
    edges = np.arange(-12.4, -8.6, 0.2); mid = 0.5 * (edges[1:] + edges[:-1])
    med = [np.median(lgo[(lgb >= a) & (lgb < b)]) if np.sum((lgb >= a) & (lgb < b)) > 10 else np.nan for a, b in zip(edges[:-1], edges[1:])]
    ax.plot(mid, med, "o", color="k", ms=5, label="binned median")
    xs = np.logspace(-12.5, -8.5, 200)
    ax.plot(np.log10(xs), np.log10(xs), "--", color="gray", label="Newton, $g=g_{\\rm bar}$")
    ax.plot(np.log10(xs), np.log10(aw.rar(xs, cH0 / (2 * np.pi))), color=PALETTE[1], lw=2.5, label=r"Hypothesis M, $\kappa'=1/2\pi$")
    ax.set_xlim(-12.5, -8.5); ax.set_ylim(-12.5, -8.5)
    ax.set_xlabel(r"$\log_{10} g_{\rm bar}$ [m s$^{-2}$]"); ax.set_ylabel(r"$\log_{10} g_{\rm obs}$ [m s$^{-2}$]")
    ax.set_title(f"All {lgb.size} SPARC points (nominal mass-to-light ratios)"); ax.legend(loc="upper left")
    fig.colorbar(hb, ax=ax, label="points per cell")
    plt.tight_layout(); plt.show()

In [ ]:
if HAVE_SPARC:
    import time
    fits = {}
    for floor in (0.0, 0.05):
        t0 = time.time()
        f = aw.fit_sample(gals, err_floor=floor)
        f["hom"] = aw.homogeneity(f["log_mu"], f["log_sd"], ~f["at_edge"])
        fits[floor] = f
        G, h = f["glob"], f["hom"]
        print(f"error floor {floor:>3.0%}: kappa' = {G['kappa']:.3f} (bootstrap 68%: {G['boot_lo']:.3f}-{G['boot_hi']:.3f}), "
              f"g0 = {G['kappa'] * cH0:.2e} m/s^2 | {h['n']} galaxies: scatter {h['tau_dex']:.2f} dex, "
              f"chi2 {h['chi2']:.0f}/{h['dof']}  [{time.time() - t0:.0f} s]")
    print("paper, Table 3:   0.189 (0.158-0.204), scatter 0.41 dex;   0.158 (0.142-0.164), scatter 0.34 dex, chi2 1935/147")

In [ ]:
if HAVE_SPARC:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.4]})
    for (floor, f), col in zip(fits.items(), PALETTE):
        k, tot, G = f["kappas"], f["total"], f["glob"]
        a1.plot(k, 2 * (tot.max() - tot), color=col, lw=2.5, label=f"error floor {floor:.0%}")
        a1.axvspan(G["boot_lo"], G["boot_hi"], color=col, alpha=0.15)
    a1.axvline(1 / (2 * np.pi), color="k", ls="--", label=r"$1/2\pi$")
    a1.set_xlim(0.1, 0.25); a1.set_ylim(0, 300)
    a1.set_xlabel(r"$\kappa'=g_0/(cH_0)$"); a1.set_ylabel(r"$-2\,\Delta\ln L$ (all galaxies)")
    a1.set_title("Common scale: shaded = bootstrap 68%"); a1.legend(loc="upper left")
    a1.text(0.98, 0.03, "quoted errors: a few galaxies with very\nsmall errors make the 0% curve jagged", transform=a1.transAxes, ha="right", fontsize=8, color=PALETTE[0])

    f = fits[0.05]; m = ~f["at_edge"]
    mu, sd = f["log_mu"][m], f["log_sd"][m]
    o = np.argsort(mu); kap = np.exp(mu[o])
    a2.errorbar(np.arange(m.sum()), kap, yerr=[kap - np.exp(mu[o] - sd[o]), np.exp(mu[o] + sd[o]) - kap],
                fmt="o", ms=2.5, color=PALETTE[0], elinewidth=0.8, label="individual galaxies (5% floor)")
    kc, tau = f["hom"]["mean_kappa"], f["hom"]["tau_dex"]
    a2.axhspan(kc * 10 ** -tau, kc * 10 ** tau, color=PALETTE[2], alpha=0.15, label=f"intrinsic scatter ±{tau:.2f} dex")
    a2.axhline(1 / (2 * np.pi), color="k", ls="--", label=r"$1/2\pi$")
    a2.set_yscale("log"); a2.set_xlabel("galaxy (sorted by fitted $\\kappa'$)"); a2.set_ylabel(r"$\kappa'$")
    a2.set_title("Galaxy by galaxy, the scale scatters"); a2.legend(fontsize=9, loc="upper left")
    plt.tight_layout(); plt.show()

In [ ]:
if HAVE_SPARC:
    by_name = {g.name: g for g in gals}
    pick = [n for n in ("NGC2403", "NGC3198", "DDO154", "UGC02885") if n in by_name]
    pick += [g.name for g in sorted(gals, key=lambda g: -len(g.R)) if g.name not in pick][: 4 - len(pick)]
    kfit = fits[0.05]["glob"]["kappa"]
    fig, axs = plt.subplots(1, 4, figsize=(15, 3.8))
    for ax, n in zip(axs, pick):
        g = by_name[n]; Rm = g.R * K.KPC
        vb2 = (np.abs(g.Vgas) * g.Vgas + ups_d * np.abs(g.Vdisk) * g.Vdisk + ups_b * np.abs(g.Vbul) * g.Vbul) * 1e6
        gbar = np.maximum(vb2, 1e-6) / Rm
        ax.errorbar(g.R, g.Vobs, g.eV, fmt="o", color="k", ms=3, capsize=1.5, label="observed")
        ax.plot(g.R, np.sqrt(np.maximum(vb2, 0)) / 1e3, "--", color="gray", label="baryons (Newton)")
        ax.plot(g.R, np.sqrt(Rm * aw.rar(gbar, kfit * cH0)) / 1e3, color=PALETTE[1], lw=2.2, label=fr"Hypothesis M, $\kappa'={kfit:.3f}$")
        ax.set_title(n); ax.set_xlabel("R [kpc]")
    axs[0].set_ylabel("rotation speed [km/s]"); axs[0].legend(fontsize=8, loc="lower right")
    fig.suptitle("Examples at the common scale, catalogue distance and inclination, nominal mass-to-light ratios", y=1.02)
    plt.tight_layout(); plt.show()

**Reading the figures.** The common scale lands next to the candidate: with the $5\%$ floor, $\kappa'=0.158$ against $1/2\pi=0.159$. That is the empirical acceleration scale of the radial acceleration relation, and its closeness to $cH_0/2\pi$ was already noted by Milgrom in 1983. The fit confirms the number; it does not by itself prefer the mechanism of Hypothesis M over other models with the same interpolation function.

The second panel is the open problem. Fitted one by one, the galaxies scatter about the common value by about $0.34$ dex, well beyond their error bars. Whether such scatter is a real variation of the scale or a sign that the data errors are underestimated is disputed for the same fits in modified Newtonian dynamics (Rodrigues et al. 2018; Kroupa et al. 2018; Li et al. 2018). So the paper's falsification criterion, a variation of $\kappa'$ between galaxies beyond their true uncertainties, is not yet decided by these data.

## 12. Co-scaling with $\gamma$: where the absolute frame could show

In the absolute-frame reading of section 2, a clock moving through the cosmic frame runs at $1/\gamma$ and the separations it assigns along its motion are $\gamma$ times the absolute ones. So every scale co-scales with $\gamma$: a clock rate as $\gamma^{n_c}$, a length along the motion as $\gamma^{n_\parallel}$, a length across it as $\gamma^{n_\perp}$.

Expanding $\gamma^n\approx1+\tfrac n2 v^2/c^2$ gives the Robertson–Mansouri–Sexl test theory of special relativity, with $(\alpha,\beta,\delta)=\tfrac12(n_c,n_\parallel,n_\perp)$. Each classic experiment measures one combination of these:

In [ ]:
nc, npar, nperp = smp.symbols(r"n_c n_\parallel n_\perp")
labels = [r"\text{clocks (Ives--Stilwell):}\quad \alpha+\tfrac12",
          r"\text{Kennedy--Thorndike:}\quad \beta-\alpha-1",
          r"\text{Michelson--Morley:}\quad \beta-\delta-\tfrac12"]
for lab, ex in zip(labels, aw.rms_test_combinations(nc, npar, nperp)):
    show(lab, smp.nsimplify(ex))
print("exponents (-1, +1, 0) give", aw.rms_test_combinations(-1, 1, 0), "-> exactly special relativity")

In [ ]:
from physics.symbolic.function_generators.paper import gamma_scaling as gs
H0_yr = K.H0 * K.YEAR
kt_lo, kt_hi = aw.allowed_window(*gs.KENNEDY_THORNDIKE)
mm_lo, mm_hi = aw.allowed_window(*gs.MICHELSON_MORLEY)
gz_lo, gz_hi = aw.allowed_window(*gs.REDSHIFT_GALILEO)
cs_lo, cs_hi = aw.allowed_window(*gs.PPN_GAMMA_CASSINI)
ll_lo, ll_hi = aw.allowed_window(*gs.GDOT_LLR_PER_YR)
d_nc = 2 * gs.IVES_STILWELL_ALPHA
d_npar = d_nc + 2 * max(abs(kt_lo), abs(kt_hi))
d_nperp = d_npar + 2 * max(abs(mm_lo), abs(mm_hi))
rows = [  # (label, allowed |departure| at 2 sigma, kind)
    (r"$n_c$  (moving clocks)", d_nc, "velocity"),
    (r"$n_\parallel$  (lengths along $v$)", d_npar, "velocity"),
    (r"$n_\perp$  (lengths across $v$)", d_nperp, "velocity"),
    (r"$n_c$  (static clocks, Galileo)", max(abs(gz_lo), abs(gz_hi)), "gravity"),
    (r"$n_R=\gamma_{\rm PPN}$  (radial length, Cassini)", max(abs(cs_lo), abs(cs_hi)), "gravity"),
    (r"$p_{\rm orbit}-p_{\rm atom}$  (cosmic, lunar ranging)", max(abs(ll_lo), abs(ll_hi)) / H0_yr, "cosmic"),
]
cols = {"velocity": PALETTE[0], "gravity": PALETTE[2], "cosmic": PALETTE[3]}
fig, ax = plt.subplots(figsize=(10, 3.8))
for i, (lab, d_, kind) in enumerate(rows):
    ax.barh(i, d_, color=cols[kind]); ax.text(d_ * 1.3, i, f"{d_:.1e}", va="center", fontsize=9)
ax.set_yticks(range(len(rows)), [r[0] for r in rows]); ax.invert_yaxis()
ax.set_xscale("log"); ax.set_xlim(1e-9, 1e-1); ax.grid(axis="y", visible=False)
ax.set_xlabel("largest allowed departure from the relativistic exponent (2σ)")
ax.set_title("Every exponent is pinned to its relativistic value")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=col, label=kind) for kind, col in cols.items()], loc="upper right")
plt.tight_layout(); plt.show()

In [ ]:
t = np.linspace(0, 1, 400)                                  # years
vd, vo = K.V_DIPOLE, gs.V_EARTH_ORBIT
v2 = vd**2 + vo**2 + 2 * vd * vo * np.cos(2 * np.pi * t)     # orbit aligned with the dipole (largest case)
kt_max = max(abs(kt_lo), abs(kt_hi))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
a1.plot(t * 12, v2 / (2 * K.C**2) * 1e7, color=PALETTE[0], lw=2.5)
a1.set_xlabel("month"); a1.set_ylabel(r"$\gamma-1$  [$10^{-7}$]")
a1.set_title(f"Earth's γ relative to the cosmic frame: annual swing {aw.annual_gamma_modulation():.2e}")
sig = aw.kt_annual_signal(kt_max) * np.cos(2 * np.pi * t)
a2.fill_between(t * 12, -np.abs(sig), np.abs(sig), color=PALETTE[1], alpha=0.25, label="still allowed")
a2.plot(t * 12, sig, color=PALETTE[1], lw=2.2, label=fr"largest allowed, $|\beta-\alpha-1|={kt_max:.1e}$")
a2.axhline(0, color="k", lw=1, label="special relativity")
a2.set_xlabel("month"); a2.set_ylabel("clock-vs-cavity fractional shift")
a2.set_ylim(-5e-14, 6e-14); a2.set_title("The direct test: an annual modulation"); a2.legend(fontsize=8, loc="upper center", ncol=3)
plt.tight_layout(); plt.show()
print(f"signal = (beta - alpha - 1) x {aw.kt_annual_signal(1.0):.2e}  (paper: 2.5e-7)")

**What this section shows.** The co-scaling of this construction is not a modification of relativity: with exponents $(-1,+1,0)$ it *is* special relativity, written in its absolute frame, and clock and cavity experiments fix those exponents to between $10^{-7}$ and $10^{-8}$. In gravity the same exponents take their general-relativistic values to about $5\times10^{-5}$, and lunar ranging requires atoms and orbits to co-scale identically over cosmic time to about $10^{-3}$ of the expansion rate. A perfectly universal co-scaling, everything changing together, cannot be seen by any measurement.

Where the absolute frame *could* show is the right-hand panel: as Earth's orbit changes its speed through the cosmic frame, any departure from these exponents would appear as a yearly clock-against-cavity modulation. Current experiments see none.

## 13. Dimensional audit and the full consistency dashboard

Every equation is checked for consistent units with an exponent algebra over (length, mass, time). The convention $\hat\tau=1$ s is made explicit wherever it was implicit. The audit records which forms were inconsistent in an earlier draft and confirms the corrected forms.

In [ ]:
rows = aw.dimension_audit()
md_rows = ["| equation | consistent | note |", "|---|:---:|---|"]
for row in rows:
    md_rows.append(f"| `{row.name}` | {'✅' if row.consistent else '❌'} | {row.note} |")
display(Markdown("\n".join(md_rows)))

Finally, every module of the paper subpackage has a self-check. (`numerology` checks the relations of a numerical appendix that has been removed from the paper; it is kept as a record.) Running all of them reproduces each verified statement in this notebook: symbolic identities, numerical integrations, and quoted numbers. The SPARC check repeats both fits of section 11.

In [ ]:
checks = {n: getattr(aw, n) for n in sorted(dir(aw)) if n.startswith("check_")}
results = {}
for name, fn in checks.items():
    rep = fn(os.path.expanduser(SPARC_DATA_DIR)) if name == "check_sparc" else fn()
    results[name.removeprefix("check_")] = (sum(ok for _, ok, _ in rep.rows), len(rep.rows))

names = list(results)
passed = [results[n][0] for n in names]; total = [results[n][1] for n in names]
fig, ax = plt.subplots(figsize=(9, 0.45 * len(names) + 1))
ax.barh(names, total, color="#e3e3e3", label="checks")
ax.barh(names, passed, color=PALETTE[2], label="passed")
for i, (p_, t_) in enumerate(zip(passed, total)):
    ax.text(t_ + 0.5, i, f"{p_}/{t_}", va="center", fontsize=9)
ax.invert_yaxis(); ax.set_xlabel("number of checks"); ax.legend(loc="lower right")
ax.set_title(f"All modules: {sum(passed)}/{sum(total)} checks pass"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
# Print any module's full report, for example the classical-simultaneity checks:
aw.check_absolute_time().finish()

## 14. What is established, assumed and open

**Established (verified above)**

* The Lorentz factor is the slope geometry $\gamma=\sqrt{1+(\partial_x y)^2}$, and the Lorentz transformation can be written with one absolute time: $c\,t_U=\gamma c\tau-y_U$. Every observer recovers the same $\tau$, the relativity of simultaneity is a displacement along $y$, and moving clocks run at $1/\gamma$ because of their own advance along $y$.
* In gravity the absolute slice is the exactly flat Painlevé–Gullstrand slice. The Flamm embedding gives the static spatial metric and the lapse $N=1/\gamma$, and its height equals the synchronization gap.
* The same flow relation links Schwarzschild and Hubble flow, with the Sciama identity at the horizon.
* Interiors (uniform density, Buchdahl, Tolman VII), frame dragging, the Sagnac gap, the ISCO and the perihelion advance all agree with general relativity.
* Clocks and lengths that co-scale with the Lorentz factor of motion relative to the absolute frame, with exponents $(-1,+1,0)$, are special relativity written in that frame. Experiments fix the exponents to between $10^{-7}$ and $10^{-8}$ (section 12).

**Assumed**

* The absolute frame is physically distinguished. The paper identifies it with the flow frame: the cosmic rest frame on large scales (measured through the CMB dipole) and the free-fall frame locally.
* The static field equations are imported from general relativity; a single five-dimensional action that yields them is still missing.
* Hypothesis M: the horizon cutoff of inertia, the interpolation function and $\kappa'$.

**Open**

* A varying $G_{44}$ is excluded by Cassini unless $p_y=(3p-\epsilon)/2$, and the origin of that stress is unknown.
* Locally, the classical-simultaneity reading has the same predictions as special relativity. A distinguishing observable must come from a sector with its own dynamics for $y$: the non-gradient potential, a varying $G_{44}$, or a derivation of Hypothesis M.
* Flat absolute slices are shown for spherical, Hubble and slowly rotating systems, not for general spacetimes.
* Fitted to SPARC, Hypothesis M gives $\kappa'\approx0.16$, next to $1/2\pi$, but individual galaxies scatter about it by more than their quoted errors. Whether that scatter is real is disputed (section 11).
* A yearly clock-against-cavity modulation is where the absolute frame could show (section 12); none has been seen.